# Mutability and why it matters

Today, we're going to introduce a new term -- _mutability_ -- and understand the consequences it has for our model of how memory works in Python. Before we can do that, we need to introduce a new data type -- the list.

## Lists

We'll look at lists properly in a few weeks (lab 5). But for now, we'll just do a very quick introduction to the basics.

A list is an ordered data structure that we can use to store a sequence of data. We can create an empty list by typing ```my_list = list()``` or `my_list = []`. We can create a non-empty lsist by enclosing a list of variables in square brackets - ```names = ['Martin', 'Arthur', 'Hemma', 'Josh']```. 

In [2]:
names = ['Martin', 'Arthur', 'Hemma', 'Josh']
print(names)
print(type(names))

['Martin', 'Arthur', 'Hemma', 'Josh']
<class 'list'>


We can access a single element of a list by using index notation as below:

In [3]:
print(names[0])
print(names[3])

Martin
Josh


We can add things to our list by using the append _method_. e.g.

In [4]:
names.append("Fred")
print(names)

['Martin', 'Arthur', 'Hemma', 'Josh', 'Fred']


All of this should seem pretty straightforward. A list of a just a collection of variables. What will the following code do:

In [5]:
a = [1, 2, 3]
b = a
a.append(4)
print(b)

[1, 2, 3, 4]


Huh? Why does that print `[1, 2, 3, 4]`? `b` was assigned the same value as `a`. We then appended to `a`, but `b` also changed? What's going on?

To answer this question, we'll need to refine our mental model of how memory actually works in Python. As part of this process, we'll see that Python actually has two categories of data type -- immutable (Ints, Floats, Strings, Tuples, etc) and mutable (Lists, Dictionaries). We'll also see that certain operations are permitted for mutable types, but not for immutable types. 

## A Better Model of Memory in Python

Previously, we told you that we can think of memory as a set of boxes, with each box holding three things:

- A name
- A value 
- A data type

and that one variable was equivalent to one box. In fact, that's not quite right. Instead, we should think of everything stored in memory as an object. 

An object has:

- An address 
- A type
- A value

An expression in Python such as `x + 1` produces an object with a value equal to whatever `x+1` is. Assigning an object to a variable gives it a name -- `y = x + 1` creates the variable y. However, this is __not__ done by adding a name to the object, but instead creating a separate table of pairs of names and objects. This table tells us which object a particular name refers to. Assignment (`x=5`) is sometimes called _binding_, because we are associating an object with a particular name. 

Last week, we spoke about the difference between _local_ and _global_ scope. Now that we've got a fuller model of how memory works in Python, you can hopefully see how scope is implemented. Each scope is defined by the name table that associates names with objects. Each function gets its own table (the local scope table), with a single global table for the entire program. Each time we refer to a variable, we first check any local name tables, before checking the global name table. When a function returns, any local name tables disappear. 

Let's run through some simple examples to see how this works before we try to understand some of the implications.

In [ ]:
x = 5
y = x
x = x + 1

When line 1 runs, we create an object (let's call it "A" ) with a value of 5 and a data type of Int. We also create an associated in our name table that the name x is bound to the object A. On the next like, we create a new variable y which is equal to x. Importantly, Python _doesnt_ do this by creating a new object with the same value as x, but instead by creating a new association in the name table that the name y also points at the object A. On line 3, we re-assign x. When this line runs, we first create a new object with value `x+1` (let's call it "B"). We then update our name table to reflect that the name x now points at object B. 

To see that this is what really happens, we can use the built-in Python function `id()` to see which memory address a name is referring to. 

x = 5
y = 5
print(id(x))
print(id(y))
x = x + 1
print(id(x))

## Rebinding

One of the consequences of this new model, is that we need to update our understanding of what happens when we write an expression like `x = x + 1`. In our prevous model of memory as a set of boxes, this line of code would read the value of the variable x, add one to 1, and then update the value of x with the new value. 

In practice, that's not what happens. Instead, when we have a line like that, we create a new object with value equal to `x + 1` and then update the name table so that `x` points at the new object. We can confirm that with the `id()` function -- as you can see below, the variable on line 3 is in a different location to the variable in line 1. 

In [10]:
x = 5
print(id(x))
x = x + 1
print(id(x))

4373588400
4373588432


We see a similar thing with mutable data types such as lists. For exampe, the code:

In [9]:
lst = [1, 2, 3]
print(id(lst))
lst = lst + [4]
print(id(lst))

4564661440
4564648448


## Mutation

Mutating a variable means changing the value of the object that a name refers to, rather than creating a new object and refering to that. While both mutable and immutable types can be _rebound_, only mutable types can be _mutated_. 

In [12]:
lst = [1, 2, 3]
print(id(lst))
lst.append(4)
print(id(lst))

4644444544
4644444544


## Aliasing

Aliasing is when two or more names refer to the same object. For immutable types, the object that is pointed at can't change, so mostly aliasing is harmless and invisible. 

For mutable types, such as a list, any instruction which changes the value of an object, will change what _both_ names evaluate to. This is why our initial example gives unexpected behaviour.

In [16]:
a = [1, 2, 3]
b = a
print(id(a))
print(id(b))
a.append(4)
print(b)
b.append(5)
print(a)

4644469312
4644469312
[1, 2, 3, 4]
[1, 2, 3, 4, 5]


As a and b both refer to the same object calling `a.append(4)` changes both a and b. The same is true in the other direction -- calling b.append(5) also changes the value of a.

An alias can be broken by rebinding a name.

In [17]:
a = [1, 2, 3]
b = a 
print(id(a))
print(id(b))
a = [9, 9]
print(a)
print(b)
print(id(a))
print(id(b))

4564633856
4564633856
[9, 9]
[1, 2, 3]
4644488064
4564633856


We can check whether two names are aliases of each other with the `is` keyword. This compares whether two names refer to the same object. 

In [18]:
a = [1, 2, 3]
b = a
print(a is b)

a = [9, 9]
print(a is b)

True
False


If we want to avoid creating an alias, we can use the `.copy()` method. This creates a second object, so the names now refer to two different things. 

In [20]:
a = [1, 2, 3]
b = a.copy()
print(id(a))
print(id(b))
print(a is b)
print(a == b)

4644445824
4644442240
False
True


## Mutability and Functions

Let's revisit what happens when we call a function. Previously, we said that when we call a function, we create a "local" variable and copy the value of the argument to that variable. In fact, that's not what happens. Instead, we create an alias -- in the code below, the variable `b` points at the same object as the variable `a` that is used as the input. 

In [21]:
a = 5

def print_var(b):
    print(id(b))

print(id(a))
print_var(a)

4373588400
4373588400


If we try to update the value of `b` within the function, we rebind `b` to the new object that we create, and so `a` remains unchanged.

In [22]:
a = 5

def print_var(b):
    b = b + 1
    print(id(b))

print(id(a))
print_var(a)

4373588400
4373588432


However, although our previous model was wrong, it still predicted the same behaviour as our new model -- in both cases, `a` is unchanged by whatever happens inside the function. This is the case for immutable data types. However, for mutable data types, we see that the new model gets the behaviour correct, whilst the old model doesn't. For example, 

In [23]:
a = [1, 2, 3]

def print_var(b):
    b.append(4)
    print(id(b))

print_var(a)
print(a)
print(id(a))

4644515712
[1, 2, 3, 4]
4644515712


Here, our function call again makes an alias to the object refered to by `a`. However, as a `a` is a list (and thus mutable), when we modify the object within the function, we also modify the object that a points to. 

So to recap, _rebinding_ a name within a function has no effect outside of the function. However, mutating the value of an object inside a function _does_ have an effect outside the function. This is one of the reasons why mutability matters!


### Function Scope Revisited

Last week, we said that you can read a global variable from inside a function, but you can't change it. 

In [24]:
number = 5

def my_function(val):
    number = number + val
    return number 

my_function(number)

UnboundLocalError: local variable 'number' referenced before assignment

More correctly, we should say that you can't _rebind_ a global name from within a function. But you _can_ mutate an object that is referred to by a name defined at global scope (provided the object is mutable of course!)

In [26]:
waypoints = []

def add_waypoint(x, y):
    waypoints.append((x, y))

add_waypoint(1, 1)
print(waypoints)

[(1, 1)]


In [28]:
waypoints = []

def add_waypoint_v2(x, y):
    waypoints = waypoints + [[x, y]]

add_waypoint_v2(1, 1)

UnboundLocalError: local variable 'waypoints' referenced before assignment

### Returning from a function

Stuff

### Good Practice

One consequence of how functions behave with mutable types is that it's easy to confuse ourselves if a function is *both* mutating its input and returning something. By convention, a function in Python shoulder either:

1. Mutate its input and return None
2. Leave its input unchanged and return a new object. 

One example of this is that Python has two ways of sorting a list. ```lst.sort()`` will sort the list _in-place_ and return None. On the other hand, sorted(lst) will return a new list with `lst` unchanged. 

In [31]:
lst = [5, 2, 3]
print(id(lst))
val = lst.sort()
print(val)
print(id(lst))

4645973056
None
4645973056


In [32]:
lst = [5, 2, 3]
print(id(lst))
val = sorted(lst)
print(val)
print(id(lst))
print(id(val))

4645986688
[2, 3, 5]
4645986688
4645478592


## Copying Mutable Data Types

Stuff. 

## Recap

Stuff!

## Examples 